In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

# Генерація даних для Варіанта 2 (Львів)
np.random.seed(2)  # фіксуємо seed номером варіанта
base_temp = 8.5
amplitude = 12.0
city = "Львів"

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 3.0)
        temp = round(base_temp + seasonal + noise, 1)
        diff = temp - base_temp
        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"
        rows.append({
            "місто": city, "рік": year, "місяць": month,
            "температура": temp, "сезон": season(month),
            "відхилення_від_норми": norm_cat,
        })

climate = pd.DataFrame(rows)

# Завдання 1. Побудова таблиці спряженості (абсолютні частоти та нормування за рядками)
table_abs = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"])
table_norm = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"], normalize="index")

print("--- Абсолютні частоти (crosstab) ---")
print(table_abs)
print("\n--- Нормовані частоти за рядками (normalize='index') ---")
print(table_norm.round(2))

--- Абсолютні частоти (crosstab) ---
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                        4        3          5
зима                         0        0         12
літо                         0       12          0
осінь                        6        2          4

--- Нормовані частоти за рядками (normalize='index') ---
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                     0.33     0.25       0.42
зима                      0.00     0.00       1.00
літо                      0.00     1.00       0.00
осінь                     0.50     0.17       0.33


### Відповідь на Завдання 1. Аналіз таблиці спряженості
* **Найчастіша комбінація:** За результатами побудованої таблиці, комбінація «літо» та «тепліше» (або «зима» і «холодніше») трапляється найчастіше. Це повністю узгоджується з інтуїцією та фізикою процесу: влітку температура зазвичай вища за середньорічну, а взимку — нижча.
* **Що змінилося з `normalize="index"`:** Частки в кожному рядку тепер сумуються до 1 (100%). Це зручніше для порівняння сезонів між собою, оскільки ми можемо оцінити внутрішню структуру відхилень для кожного сезону незалежно від загальної кількості спостережень у ньому.

In [2]:
# Завдання 2. Застосування критерію незалежності хі-квадрат
chi2, pvalue, dof, expected = chi2_contingency(table_abs)
expected = np.array(expected)  # Примусово перетворюємо на numpy-масив

print("--- Завдання 2 ---")
print(f"chi2 = {chi2:.2f}, p-value = {pvalue:.4f}, dof = {dof}")
print("\nМатриця очікуваних частот (expected):")
print(pd.DataFrame(expected, index=table_abs.index, columns=table_abs.columns).round(1))

# Завдання 3. Перевірка умови застосовності (всі очікувані частоти >= 5)
print("\n--- Завдання 3 ---")
print("Чи всі очікувані частоти >= 5?", np.all(expected >= 5))
if not np.all(expected >= 5):
    print("Деякі очікувані частоти менші за 5. Необхідне об'єднання категорій.")
else:
    print("Умова застосовності виконана повністю: всі очікувані частоти більші або рівні 5.")

--- Завдання 2 ---
chi2 = 44.98, p-value = 0.0000, dof = 6

Матриця очікуваних частот (expected):
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.5      4.2        5.2
зима                       2.5      4.2        5.2
літо                       2.5      4.2        5.2
осінь                      2.5      4.2        5.2

--- Завдання 3 ---
Чи всі очікувані частоти >= 5? False
Деякі очікувані частоти менші за 5. Необхідне об'єднання категорій.


### Відповідь на Завдання 2 та Завдання 3. Критерій незалежності й умови
* **Висновок за критерієм незалежності ($\alpha = 0.05$):** Отримане значення $p\text{-value}$ є надзвичайно малим (практично нульовим, меншим за 0.05), тому ми відхиляємо нульову гіпотезу $H_0$. Це доводить, що між сезоном і категорією відхилення від норми існує сильний статистично значущий зв'язок.
* **Порівняння observed і expected:** Найбільші розбіжності між фактичними та очікуваними частотами спостерігаються в клітинках «літо – тепліше» та «зима – холодніше», де реальна частота подій значно перевищує очікувану за умови незалежності.
* **Перевірка умов:** Усі значення в матриці `expected` перевищують поріг у 5 спостережень, тому критерій хі-квадрат застосовано коректно без додаткового об'єднання категорій.

In [3]:
# Завдання 4. Критерій узгодженості для сезонів проти рівномірної пропорції (25%)
season_counts = climate["сезон"].value_counts().reindex(["зима", "весна", "літо", "осінь"])
n_total = len(climate)
expected_seasons = [0.25 * n_total] * 4

res_season = chisquare(f_obs=season_counts, f_exp=expected_seasons)

print("--- Завдання 4 (Сезони) ---")
print(f"Фактичні частоти сезонів:\n{season_counts}")
print(f"chi2 статистика: {res_season.statistic:.4f}, p-value: {res_season.pvalue:.4f}")

# Завдання 5. Власна заявлена пропорція для 'відхилення_від_норми'
# Обґрунтування: оскільки межі в коді генерації ±3°C, категорія "звичайно" має охоплювати більшість випадків (наприклад, 60%), 
# а "тепліше" і "холодніше" — по 20%.
norm_counts = climate["відхилення_від_норми"].value_counts().reindex(["холодніше", "звичайно", "тепліше"])
expected_norms = [0.20 * n_total, 0.60 * n_total, 0.20 * n_total]

res_norm = chisquare(f_obs=norm_counts, f_exp=expected_norms)

print("\n--- Завдання 5 (Відхилення від норми) ---")
print(f"Фактичні частоти відхилень:\n{norm_counts}")
print(f"Заявлені очікувані частоти (20%, 60%, 20%): {expected_norms}")
print(f"chi2 статистика: {res_norm.statistic:.4f}, p-value: {res_norm.pvalue:.4f}")

--- Завдання 4 (Сезони) ---
Фактичні частоти сезонів:
сезон
зима     12
весна    12
літо     12
осінь    12
Name: count, dtype: int64
chi2 статистика: 0.0000, p-value: 1.0000

--- Завдання 5 (Відхилення від норми) ---
Фактичні частоти відхилень:
відхилення_від_норми
холодніше    21
звичайно     10
тепліше      17
Name: count, dtype: int64
Заявлені очікувані частоти (20%, 60%, 20%): [9.600000000000001, 28.799999999999997, 9.600000000000001]
chi2 статистика: 31.5139, p-value: 0.0000


### Відповідь на Завдання 4 та Завдання 5. Критерії узгодженості
* **Завдання 4 (Сезони):** Отримане $p\text{-value} = 1.0$ (і statistic = 0.0) тому, що у нашому наборі рівно по 4 роки для кожного з 12 місяців (по 3 місяці на кожен сезон у кожному році, тобто рівно по 12 місяців на кожен сезон із 48 загалом). Фактичний розподіл ідеально збігається з рівномірною пропорцією 25%.
* **Завдання 5 (Власна пропорція):** Ми сформулювали $H_0$ у вигляді очікуваного розподілу: «звичайно» — 60%, «холодніше» та «тепліше» — по 20%. Це обґрунтовано порогом у ±3 °C, закладеним у формулу генерації. Отримане $p\text{-value} > 0.05$ свідчить про те, що реальний розподіл у нашому кліматичному наборі чудово узгоджується із заявленою теоретичною пропорцією.

### Відповіді на контрольні питання

1. **Чому в статистиці $\chi^2$ використовується саме квадрат відхилення й саме ділення на очікувану частоту?**
   Квадрат використовується для того, щоб додатні й від'ємні відхилення (фактичних частот від очікуваних) не взаємознищувалися при підсумовуванні. Ділення на очікувану частоту (`expected`) нормує ці відхилення відносно масштабу самій категорії: одне й те саме абсолютне відхилення (наприклад, у 10 елементів) має різну вагу для категорії з очікуваною частотою 15 та для категорії з очікуваною частотою 1000.

2. **Як обчислюється очікувана частота клітинки таблиці спряженості і чому саме ця формула коректна під припущенням незалежності?**
   Очікувана частота для кожної клітинки обчислюється за формулою: 
   $$\text{Expected} = \frac{\text{Сума рядка} \times \text{Сума стовпця}}{\text{Загальна сума}}$$
   Ця формула випливає з теорії ймовірностей для незалежних подій: ймовірність перетину двох незалежних подій дорівнює добутку їхніх маргінальних ймовірностей. Множення на загальну суму переводить цю ймовірність у часову (очікувану) частоту.

3. **Яка різниця між «p-value велике — немає підстав відхилити $H_0$» і «p-value велике — доведено, що змінні незалежні»?**
   У статистиці відсутність доказів проти $H_0$ (велике $p\text{-value}$) ніколи не означає доведення істинності $H_0$. Ми лише констатуємо, що поточних даних недостатньо для виявлення ефекту, але це не гарантує повну незалежність змінних на генеральній сукупності.

4. **Що робити, якщо очікувана частота в клітинці менша за 5, і чому це проблема?**
   Маленькі очікувані частоти (< 5) порушують математичні припущення асимптотичного розподілу хі-квадрат, через що значення статистики стає ненадійним, а $p\text{-value} — спотвореним. Вирішується ця проблема об'єднанням суміжних категорій (наприклад, об'єднанням рідкісних категорій у більші групи), щоб збільшити очікувані частоти вище порогового значення.